In [24]:
# ## install finrl library
!pip install wrds
!pip install swig
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install finrl



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to c:\users\email\appdata\local\temp\pip-install-xgwcz0ci\elegantrl_28833741bfa74ba1bc8de086d9d69126
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 3bdc958c8e624b61a9e661832b01fef816924f61
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git 'C:\Users\email\AppData\Local\Temp\pip-install-xgwcz0ci\elegantrl_28833741bfa74ba1bc8de086d9d69126'

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import warnings
warnings.filterwarnings("ignore")

In [26]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

import gym
from finrl import config
from finrl.agents.stablebaselines3.models import DRLEnsembleAgent
from finrl.meta.preprocessor.preprocessors import FeatureEngineer

from pprint import pprint

import sys
sys.path.append("../FinRL-Library")

import itertools

In [27]:
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

In [28]:
TRAIN_START_DATE = '2017-01-01'
TRAIN_END_DATE = '2021-10-01'
TEST_START_DATE = '2021-10-01'
TEST_END_DATE = '2023-03-01'

In [29]:
df = pd.read_csv("C:\\Users\\email\\Downloads\\historical_data.csv")

In [30]:
#Data Exploration
print("First 5 rows of the dataset:")
print(df.head())

First 5 rows of the dataset:
         date      close       high        low       open  volume     tic  day
0  2004-01-01   0.105000   0.105000   0.105000   0.105000       0  DYA.TO    3
1  2004-01-01   1.378000   1.378000   1.378000   1.378000       0  EDT.TO    3
2  2004-01-01   0.300000   0.300000   0.300000   0.300000       0  TSK.TO    3
3  2004-01-02   0.489625   0.489625   0.489625   0.489625       0  AAB.TO    4
4  2004-01-02  22.070263  22.211644  21.817266  21.899118  413000  ABX.TO    4


In [31]:
print(df.isnull().sum())  # Shows the count of missing values per column

date      0
close     0
high      0
low       0
open      0
volume    0
tic       0
day       0
dtype: int64


In [32]:
# Convert 'date' column to datetime if it is not already
df['date'] = pd.to_datetime(df['date']) 

# *Step 1: Data Cleaning*
df = df.sort_values(["date", "tic"], ignore_index=True)

# Factorize the date index
df.index = df.date.factorize()[0]
df

,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3
0,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3
0,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3
1,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4
1,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4
...,...,...,...,...,...,...,...,...
5280,2024-12-30,20.230000,20.309999,20.150000,20.309999,700,ZWS.TO,0
5280,2024-12-30,53.099998,53.360001,52.830002,53.299999,11600,ZWT.TO,0
5280,2024-12-30,10.510000,10.520000,10.430000,10.500000,168700,ZWU.TO,0
5280,2024-12-30,42.599998,42.610001,42.599998,42.610001,300,ZXM.TO,0


In [33]:

# Drop rows where any of the required columns have NaN values
df = df.dropna(subset=["close", "high", "low", "open", "tic", "day"])

# *Step 3: Add User Defined Features*
df["daily_return"] = df.close.pct_change(1)

In [34]:
df.head()

,date,close,high,low,open,volume,tic,day,daily_return
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3,NaN
0,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3,12.123810
0,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3,-0.782293
1,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4,0.632084
1,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4,44.075824


In [35]:
INDICATORS = ['macd',
               'rsi_30',
               'cci_30',
               'dx_30']

In [36]:
#TICKERS = ["RY.TO", "TD.TO", "BNS.TO", "ENB.TO", "SHOP.TO"]  # Example TSX tickers

In [37]:
#df = df[df['tic'].isin(TICKERS)]
df.shape

(4066965, 9)

In [38]:
feature_engineer = FeatureEngineer(use_technical_indicator=True, tech_indicator_list=INDICATORS)


In [39]:
df = feature_engineer.add_technical_indicator(df)

In [40]:
df = feature_engineer.add_turbulence(df)

In [41]:
df["date"] = pd.to_datetime(df["date"])  # Convert main dataset date column

In [ ]:
stock_dimension = len(df.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")



Stock Dimension: 1631, State Space: 9787


In [43]:
env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "print_verbosity":5

}


In [44]:
rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

ensemble_agent = DRLEnsembleAgent(df,
                 train_period=(TRAIN_START_DATE,TRAIN_END_DATE),
                 val_test_period=(TEST_START_DATE,TEST_END_DATE),
                 rebalance_window=rebalance_window,
                 validation_window=validation_window,
                 **env_kwargs)

In [45]:


DDPG_model_kwargs = {
                      #"action_noise":"ornstein_uhlenbeck",
                      "buffer_size": 10_000,
                      "learning_rate": 0.0005,
                      "batch_size": 64
                    }

TD3_model_kwargs = {"batch_size": 100, "buffer_size": 1000000, "learning_rate": 0.0001}




timesteps_dict = {'a2c' : 0,
                 'ppo' : 0,
                 'ddpg' : 10_000,
                 'sac' : 0,
                 'td3' : 0
                 }

In [46]:
df_summary = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs={},  # Not using A2C
    PPO_model_kwargs={},  # Not using PPO
    DDPG_model_kwargs=DDPG_model_kwargs,  # Using DDPG
    SAC_model_kwargs={},  # Not using SAC
    TD3_model_kwargs={},  # Not using TD3
    timesteps_dict=timesteps_dict
)

============Start Ensemble Strategy============
turbulence_threshold:  5051557442.335041
======Model training from:  2017-01-01 to  2021-10-04 00:00:00
======a2c Training========
{}
Using cpu device


ValueError: could not broadcast input array from shape (5737,) into shape (9787,)

In [47]:
df_summary

NameError: name 'df_summary' is not defined

In [ ]:
processed=df

In [ ]:
unique_trade_date = processed[(processed.date > TEST_START_DATE)&(processed.date <= TEST_END_DATE)].date.unique()

In [ ]:
import pandas as pd

df_account_value = pd.DataFrame()

for i in range(rebalance_window + validation_window, len(unique_trade_date) + 1, rebalance_window):
    temp = pd.read_csv('results/account_value_trade_{}_{}.csv'.format('ensemble', i))
    
    # Fix: Use pd.concat() instead of .append()
    df_account_value = pd.concat([df_account_value, temp], ignore_index=True)

# Compute Sharpe Ratio
sharpe = (252 ** 0.5) * df_account_value.account_value.pct_change(1).mean() / df_account_value.account_value.pct_change(1).std()
print('Sharpe Ratio:', sharpe)

# Fix any potential DataFrame joining issues
df_account_value = df_account_value.merge(df_trade_date[validation_window:].reset_index(drop=True), left_index=True, right_index=True, how='left')


In [ ]:
df_account_value.head()

In [ ]:
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline
print("==============Get Backtest Results===========")
now = datetime.datetime.now().strftime('%Y%m%d-%Hh%M')

perf_stats_all = backtest_stats(account_value=df_account_value)
perf_stats_all = pd.DataFrame(perf_stats_all)

In [ ]:
#baseline stats
print("==============Get Baseline Stats===========")
df_dji_ = get_baseline(
        ticker="RY.TO",
        start = df_account_value.loc[0,'date'],
        end = df_account_value.loc[len(df_account_value)-1,'date'])

stats = backtest_stats(df_dji_, value_col_name = 'close')